# Face Emotion Recognition — SVM RBF

**Pipeline**

`Ảnh đầu vào + 478 landmarks`
→ `Tính góc nghiêng từ landmark mắt trái/phải`
→ `Xoay landmark`
→ `Lấy mũi làm gốc`
→ `Chia theo khoảng cách 2 mắt`
→ `Flatten thành vector 1D`
→ `Chia Train / Validation / Test`
→ `StandardScaler (fit trên Train, transform Test)`
→ `SVM Classifier (RBF Kernel)`
→ `Train / tối ưu C, gamma`
→ `Lưu checkpoint (.pkl)`
→ `Predict`
→ `Cảm xúc`

In [1]:
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

RANDOM_STATE = 42
N_LANDMARKS = 478

# Đổi path này nếu notebook nằm ở vị trí khác.
CSV_PATH = Path("Output/face_landmarks.csv")

if not CSV_PATH.exists():
    for p in [Path("../Output/face_landmarks.csv"), Path("/mnt/data/face_landmarks.csv")]:
        if p.exists():
            CSV_PATH = p
            break

print("CSV:", CSV_PATH.resolve())


CSV: C:\Users\LENOVO\Documents\Study\ML\Project\yolov10-mediapipe-gesture-emotion\8 label\Output\face_landmarks.csv


## 1. Đọc CSV

In [2]:
df = pd.read_csv(CSV_PATH)

X_raw = df.drop(columns=["label"]).astype(np.float32)
y = df["label"].copy()

print("Dataset:", df.shape)
print("Feature count:", X_raw.shape[1])
print("Labels:", sorted(y.unique().tolist()))
print("\nLabel counts:")
print(y.value_counts().sort_index())


Dataset: (15061, 1435)
Feature count: 1434
Labels: [0, 2, 3, 4, 5, 6, 7, 8]

Label counts:
label
0    1710
2    1348
3    2716
4    1883
5    1788
6    2614
7    1176
8    1826
Name: count, dtype: int64


## 2. Reshape thành 478 landmark × 3 tọa độ

In [6]:
EXPECTED_FEATURES = 478 * 3

if X_raw.shape[1] != EXPECTED_FEATURES:
    raise ValueError(
        f"CSV có {X_raw.shape[1]} features. "
        f"Pipeline này cần {EXPECTED_FEATURES} features = 478 x 3."
    )

landmarks = X_raw.to_numpy(np.float32).reshape(-1, 478, 3)
print("Landmarks:", landmarks.shape)


Landmarks: (15061, 478, 3)


## 3. Tính góc nghiêng từ mắt trái/phải

Dùng:
- landmark `33` = mắt trái
- landmark `263` = mắt phải

Góc roll được tính trong mặt phẳng XY bằng `atan2(dy, dx)`.


In [7]:
LEFT_EYE = 33
RIGHT_EYE = 263
NOSE = 1

def roll_angle(face):
    left = face[LEFT_EYE]
    right = face[RIGHT_EYE]
    dx = right[0] - left[0]
    dy = right[1] - left[1]
    return np.arctan2(dy, dx)

angles = np.array([roll_angle(f) for f in landmarks], dtype=np.float32)

print("Mean angle (deg):", np.degrees(angles).mean())
print("Std angle  (deg):", np.degrees(angles).std())


Mean angle (deg): -0.58671427
Std angle  (deg): 7.071635


## 4. Xoay landmark về tư thế chuẩn

In [8]:
def rotate_z(face, angle):
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ], dtype=np.float32)
    return face @ R.T

rotated = np.empty_like(landmarks)

for i, face in enumerate(landmarks):
    rotated[i] = rotate_z(face, -roll_angle(face))

print(rotated.shape)


(15061, 478, 3)


## 5. Lấy mũi làm gốc + chia theo khoảng cách hai mắt

Sau khi xoay:
1. Dời landmark mũi về `(0, 0, 0)`.
2. Tính khoảng cách giữa mắt trái và mắt phải.
3. Chia tất cả landmark cho khoảng cách đó để chuẩn hóa scale.


In [9]:
centered = rotated - rotated[:, NOSE:NOSE+1, :]

eye_distance = np.linalg.norm(
    centered[:, RIGHT_EYE] - centered[:, LEFT_EYE],
    axis=1
)

if np.any(eye_distance <= 1e-8):
    raise ValueError("Có sample có eye distance quá nhỏ.")

normalized = centered / eye_distance[:, None, None]

print("Normalized:", normalized.shape)
print("Eye distance range:", eye_distance.min(), "→", eye_distance.max())


Normalized: (15061, 478, 3)
Eye distance range: 0.100993484 → 1.0757569


## 6. Flatten thành vector 1D

In [10]:
X = normalized.reshape(len(normalized), -1).astype(np.float32)

print("Final X:", X.shape)
print("y:", y.shape)


Final X: (15061, 1434)
y: (15061,)


## 7. Chia Train / Validation / Test


In [11]:
# Chia đúng 70% Train / 15% Validation / 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

print(f"Train ratio: {len(X_train)/len(X):.2%}")
print(f"Val ratio  : {len(X_val)/len(X):.2%}")
print(f"Test ratio : {len(X_test)/len(X):.2%}")

Train: (10542, 1434)
Val  : (2259, 1434)
Test : (2260, 1434)
Train ratio: 70.00%
Val ratio  : 15.00%
Test ratio : 15.01%


## 8. StandardScaler


In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape)
print("Val  :", X_val_scaled.shape)
print("Test :", X_test_scaled.shape)

Train: (10542, 1434)
Val  : (2259, 1434)
Test : (2260, 1434)


## 9. SVM Classifier — RBF Kernel

- `C`: trade-off giữa margin và lỗi phân loại.
- `gamma`: độ ảnh hưởng của một sample trong RBF.


In [13]:
svm_model = SVC(
    kernel="rbf",
    C=10.0,
    gamma="scale",
    probability=True,
    cache_size=4000,
    random_state=RANDOM_STATE
)

t0 = time.time()
svm_model.fit(X_train_scaled, y_train)
print(f"Training time: {time.time()-t0:.2f}s")
print("Support vectors:", svm_model.n_support_)
print("Total support vectors:", len(svm_model.support_))


Training time: 2831.93s
Support vectors: [1150  935 1789  886 1083 1793  819 1179]
Total support vectors: 9634


## 10. Tối ưu margin / support vectors bằng validation nội bộ

Test không được dùng để chọn hyperparameter.

Ta chia **Train** thành `train_inner` + `validation`, thử một grid nhỏ của `C` và `gamma`, sau đó fit lại model tốt nhất trên toàn bộ Train.


In [14]:
# Dùng Validation 15% để tối ưu C và gamma.
C_values = [1, 10, 50]
gamma_values = ["scale", 0.001, 0.01]

tuning = []

for C in C_values:
    for gamma in gamma_values:
        model = SVC(
            kernel="rbf",
            C=C,
            gamma=gamma,
            cache_size=4000,
            random_state=RANDOM_STATE
        )

        t0 = time.time()
        model.fit(X_train_scaled, y_train)
        val_pred = model.predict(X_val_scaled)

        tuning.append({
            "C": C,
            "gamma": gamma,
            "Val Accuracy": accuracy_score(y_val, val_pred),
            "Train Time (s)": time.time() - t0,
            "Support Vectors": len(model.support_)
        })

tuning_df = pd.DataFrame(tuning).sort_values(
    "Val Accuracy", ascending=False
).reset_index(drop=True)

display(tuning_df)

,C,gamma,Val Accuracy,Train Time (s),Support Vectors
0,10,0.01,0.530766,583.541686,9427
1,50,0.01,0.529880,576.462198,9413
2,50,0.001,0.517928,1148.611438,9600
3,50,scale,0.510403,1025.913097,9545
4,1,0.01,0.501549,404.262946,10169
5,10,0.001,0.478088,472.734015,9653
6,10,scale,0.461709,374.656193,9634
7,1,0.001,0.404161,203.507207,9825
8,1,scale,0.401948,190.153011,9854


## 11. Fit SVM cuối cùng với best C/gamma

In [15]:
best_C = float(tuning_df.loc[0, "C"])
best_gamma = tuning_df.loc[0, "gamma"]

# fit lại trên toàn bộ 85% Train + Validation
X_train_val_scaled = np.vstack([X_train_scaled, X_val_scaled])
y_train_val = pd.concat([
    y_train.reset_index(drop=True),
    y_val.reset_index(drop=True)
], ignore_index=True)

final_svm = SVC(
    kernel="rbf",
    C=best_C,
    gamma=best_gamma,
    probability=True,
    cache_size=4000,
    random_state=RANDOM_STATE
)

t0 = time.time()
final_svm.fit(X_train_val_scaled, y_train_val)

print(f"Final training time: {time.time() - t0:.2f}s")
print("Best C:", best_C)
print("Best gamma:", best_gamma)
print("Total support vectors:", len(final_svm.support_))

Final training time: 3746.07s
Best C: 10.0
Best gamma: 0.01
Total support vectors: 11246


## 12. Predict → Cảm xúc

In [20]:
y_pred = final_svm.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")


Accuracy : 0.5987
Precision: 0.5932
Recall   : 0.5987
F1       : 0.5908


## 13. Classification Report + Confusion Matrix

In [3]:
emotion_map = {
    0: "Anger",
    2: "Disgust",
    3: "Fear",
    4: "Happy",
    5: "Neutral",
    6: "Sad",
    7: "Shy",
    8: "Surprise"
}

labels = sorted(y.unique())
display_labels = [emotion_map[label] for label in labels]

cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=display_labels,
    yticklabels=display_labels
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("SVM - Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(classification_report(
    y_test,
    y_pred,
    labels=labels,
    target_names=display_labels,
    zero_division=0
))

NameError: name 'y_test' is not defined

## 14. Lưu checkpoint `face_emotion_svm.pkl`

Checkpoint chứa cả `SVM` và `StandardScaler` cùng metadata preprocessing.

Khi nhận ảnh mới, phải trích xuất 478 landmarks và đi qua **đúng pipeline preprocessing** trước khi predict.


In [18]:
checkpoint = {
    "model": final_svm,
    "scaler": scaler,
    "n_landmarks": 478,
    "left_eye_landmark": LEFT_EYE,
    "right_eye_landmark": RIGHT_EYE,
    "nose_landmark": NOSE,
    "kernel": "rbf",
    "C": float(best_C),
    "gamma": best_gamma,
    "labels": sorted(y.unique().tolist()),
    "preprocessing": {
        "1": "calculate roll angle from eye landmarks 33 and 263",
        "2": "rotate landmarks around Z axis",
        "3": "nose landmark 1 as origin",
        "4": "divide by inter-eye distance",
        "5": "flatten 468x3 to 1404 features",
        "6": "StandardScaler fit on train only"
    }
}

MODEL_PATH = "face_emotion_svm.pkl"
joblib.dump(checkpoint, MODEL_PATH, compress=3)

print("Saved:", MODEL_PATH)


Saved: face_emotion_svm.pkl


## 15. Inference function

In [19]:
def preprocess_single_face(face_478x3, checkpoint):
    face = np.asarray(face_478x3, dtype=np.float32)

    if face.shape != (478, 3):
        raise ValueError(f"Expected (478, 3), got {face.shape}")

    le = checkpoint["left_eye_landmark"]
    re = checkpoint["right_eye_landmark"]
    nose = checkpoint["nose_landmark"]

    # 1. angle
    dx = face[re, 0] - face[le, 0]
    dy = face[re, 1] - face[le, 1]
    angle = np.arctan2(dy, dx)

    # 2. rotate
    c, s = np.cos(-angle), np.sin(-angle)
    R = np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ], dtype=np.float32)

    rotated = face @ R.T

    # 3. nose as origin
    centered = rotated - rotated[nose]

    # 4. scale by inter-eye distance
    d = np.linalg.norm(centered[re] - centered[le])
    if d <= 1e-8:
        raise ValueError("Invalid inter-eye distance.")

    normalized = centered / d

    # 5. flatten
    vector = normalized.reshape(1, -1)

    # 6. scaler
    return checkpoint["scaler"].transform(vector)


def predict_emotion(face_478x3, checkpoint_path="face_emotion_svm.pkl"):
    checkpoint = joblib.load(checkpoint_path)
    X_one = preprocess_single_face(face_478x3, checkpoint)

    prediction = checkpoint["model"].predict(X_one)[0]

    result = {"label": prediction}

    if hasattr(checkpoint["model"], "predict_proba"):
        proba = checkpoint["model"].predict_proba(X_one)[0]
        result["confidence"] = float(np.max(proba))

    return result


## Final pipeline

`Image`
→ `MediaPipe Face Mesh`
→ `478 landmarks`
→ `eye-left / eye-right angle`
→ `rotate`
→ `nose = origin`
→ `/ inter-eye distance`
→ `flatten 1404`
→ `train/test`
→ `StandardScaler fit(train) / transform(test)`
→ `SVM RBF`
→ `tune C/gamma`
→ `predict`
→ `emotion`
→ `face_emotion_svm.pkl`
